ESG Analysis

IMPORT LIBRARY

In [3]:
import pandas as pd

LOAD RAW DATA

In [6]:
data = (r'D:\End to End\ESG_Raw_Data.csv')
df = pd.read_csv(data)

DATA PROFILING

In [11]:
print("\nShape:")
print(df.shape)


Shape:
(20035, 17)


In [13]:
print("\nFirst 5 Rows:")
print(df.head())


First 5 Rows:
   record_id facility_id    facility_name       city region reporting_month  \
0  ESG100000        F003       Pune Plant       Pune   West      2025-04-01   
1  ESG100001        F003       Pune Plant       Pune   West      2026-02-01   
2  ESG100002        F007  Hyderabad Plant  Hyderabad  South      2025-10-01   
3  ESG100003        F008     Jaipur Plant     Jaipur  North      2026-02-01   
4  ESG100004        F001      Delhi Plant      Delhi  North      2025-09-01   

  supplier_id supplier_type    energy_kwh      water_m3  production_units  \
0        S016     Chemicals  78319.526744  11577.376800      12159.877642   
1        S055     Logistics  45653.124662   6567.539614      11161.931864   
2        S021   Maintenance  66288.188776  12667.898881      13610.233734   
3        S023   Maintenance  64700.949451   9813.795213      11154.989344   
4        S025     Packaging  73894.063615  10808.464140      10127.709239   

   waste_tonnes  recycled_waste_tonnes  scope1_

In [18]:
print("\nData Types:")
print(df.dtypes)


Data Types:
record_id                 object
facility_id               object
facility_name             object
city                      object
region                    object
reporting_month           object
supplier_id               object
supplier_type             object
energy_kwh               float64
water_m3                 float64
production_units         float64
waste_tonnes             float64
recycled_waste_tonnes    float64
scope1_tco2e             float64
scope2_tco2e             float64
supplier_spend_inr       float64
energy_unit               object
dtype: object


In [28]:
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20035 entries, 0 to 20034
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   record_id              20035 non-null  object 
 1   facility_id            20035 non-null  object 
 2   facility_name          20035 non-null  object 
 3   city                   20035 non-null  object 
 4   region                 20035 non-null  object 
 5   reporting_month        20035 non-null  object 
 6   supplier_id            20035 non-null  object 
 7   supplier_type          20035 non-null  object 
 8   energy_kwh             20035 non-null  float64
 9   water_m3               19795 non-null  float64
 10  production_units       20035 non-null  float64
 11  waste_tonnes           20035 non-null  float64
 12  recycled_waste_tonnes  19915 non-null  float64
 13  scope1_tco2e           20035 non-null  float64
 14  scope2_tco2e           20035 non

In [37]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
record_id                  0
facility_id                0
facility_name              0
city                       0
region                     0
reporting_month            0
supplier_id                0
supplier_type              0
energy_kwh                 0
water_m3                 240
production_units           0
waste_tonnes               0
recycled_waste_tonnes    120
scope1_tco2e               0
scope2_tco2e               0
supplier_spend_inr       161
energy_unit                0
dtype: int64


In [39]:
print("\nDuplicates Full Rows (Ignoring record_id):")
print(df.drop(columns=['record_id']).duplicated().sum())


Duplicates Full Rows (Ignoring record_id):
35


In [44]:
print("\nDuplicate Record IDs:")
print(df['record_id'].duplicated().sum())


Duplicate Record IDs:
0


In [55]:
print("\nEnergy Unit Distribution:")
print(df['energy_unit'].value_counts())


Energy Unit Distribution:
energy_unit
kWh    19855
MWh      180
Name: count, dtype: int64


In [63]:
print("\nSupplier Type Distribution:")
print(df['supplier_type'].value_counts())


Supplier Type Distribution:
supplier_type
Chemicals       4051
Logistics       4026
Packaging       4016
Maintenance     3997
Raw Material    3945
Name: count, dtype: int64


In [70]:
print("\nNumerical Summary:")
print(df.describe())


Numerical Summary:
         energy_kwh      water_m3  production_units  waste_tonnes  \
count  2.003500e+04  19795.000000      20035.000000  20035.000000   
mean   6.491155e+04  11013.751603      12015.693242    180.296740   
std    2.434856e+04   2982.619983       2996.386216     55.058125   
min    1.819445e+01    100.000000        500.000000      1.000000   
25%    5.277361e+04   8980.797032       9986.756394    143.268632   
50%    6.490553e+04  10993.249585      12006.194837    180.729114   
75%    7.710062e+04  13029.540300      14015.449028    217.532591   
max    1.093544e+06  22298.540229      24417.012643    390.948669   

       recycled_waste_tonnes  scope1_tco2e  scope2_tco2e  supplier_spend_inr  
count           19915.000000  20035.000000  20035.000000        1.987400e+04  
mean              121.852254     38.029008     26.109923        4.510394e+05  
std                44.399733     11.995310      9.248523        1.501549e+05  
min                 0.472165      0.000000

CREATE CLEAN COPY

In [85]:
df_clean = df.copy()

DATA CLEANING

In [96]:
#REMOVE BUSINESS DUPLICATES
business_cols = [col for col in df_clean.columns if col != 'record_id']

df_clean = df_clean.drop_duplicates(subset=business_cols, keep='first').copy()

In [98]:
#REMOVE COMPLETELY BLANK ROWS
df_clean = df_clean.dropna(how='all').copy()

In [104]:
#CONVERT DATE
df_clean['reporting_month'] = pd.to_datetime(df_clean['reporting_month'], errors='coerce')

In [113]:
#STANDARDIZE ENERGY TO kWh
df_clean['energy_kwh_standardized'] = (df_clean['energy_kwh'].where(df_clean['energy_unit'] == 'kWh',df_clean['energy_kwh'] * 1000))

In [115]:
#CREATE ENERGY INTENSITY
df_clean['energy_intensity'] = (df_clean['energy_kwh_standardized']/ df_clean['production_units'])

In [121]:
#IQR OUTLIER CALCULATION
Q1 = df_clean['energy_kwh_standardized'].quantile(0.25)

Q3 = df_clean['energy_kwh_standardized'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)

upper_limit = Q3 + (1.5 * IQR)


In [128]:
print("IQR OUTLIER INFORMATION")
print("\nQ1:")
print(Q1)

print("\nQ3:")
print(Q3)

print("\nIQR:")
print(IQR)

print("\nLower Limit:")
print(lower_limit)

print("\nUpper Limit:")
print(upper_limit)


IQR OUTLIER INFORMATION

Q1:
53116.37504345996

Q3:
77250.09809060465

IQR:
24133.72304714469

Lower Limit:
16915.79047274292

Upper Limit:
113450.68266132168


In [130]:
#ENERGY OUTLIER FLAG
df_clean['energy_outlier_flag'] = (
    (df_clean['energy_kwh_standardized'] < lower_limit)
    |
    (df_clean['energy_kwh_standardized'] > upper_limit)
)


In [137]:
#WASTE LOGIC CHECK
df_clean['waste_logic_check'] = (
    df_clean['recycled_waste_tonnes'].isna()
    |
    (
        df_clean['recycled_waste_tonnes']
        <= df_clean['waste_tonnes']
    )
)

EXPORT CLEAN DATA

In [174]:
df_clean.to_csv(r'D:\End to End\ESG_Manufacturing_Clean_Data.csv', index=False)